# 📊 PROJET DATA MINING - DÉTECTION D'INTRUSIONS RÉSEAU
## KDD Cup 1999 - Exploratory Data Analysis (EDA)

**Étape:** 4 · EDA + 5 · Project Schema  
**Date:** Mars 2026  
**Rôle:** Analyse exploratoire des données préparées par P2

---

### Objectifs de ce notebook:
1. Statistiques descriptives globales
2. Distribution des classes (attack_category)
3. Analyse par protocole
4. Heatmap de corrélation
5. Distributions des features clés par classe
6. Boxplots des features les plus discriminantes
7. Feature importance (variance)
8. Project Schema (Flow Diagram)

---

### Fichiers nécessaires (dossier `data_prepared/`):
- `X_train_prepared.csv`
- `y_train.csv`
- `X_test_prepared.csv`
- `y_test.csv`

In [ ]:
# ============================================
# IMPORT DES BIBLIOTHÈQUES
# ============================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.style.use('ggplot')
sns.set_palette('Set2')

# Palette de couleurs pour les 5 classes
CLASS_COLORS = {
    'normal': '#4CAF50',
    'dos':    '#F44336',
    'probe':  '#FF9800',
    'r2l':    '#9C27B0',
    'u2r':    '#795548'
}
CLASS_ORDER = ['normal', 'dos', 'probe', 'r2l', 'u2r']

print('✅ Bibliothèques importées avec succès!')

## 1. CHARGEMENT DES DONNÉES PRÉPARÉES

In [ ]:
# ============================================
# CHARGEMENT
# ============================================

X_train = pd.read_csv('X_train_prepared.csv')
X_test  = pd.read_csv('X_test_prepared.csv')
y_train = pd.read_csv('y_train.csv').squeeze()
y_test  = pd.read_csv('y_test.csv').squeeze()

# Combiner train+test pour l'EDA globale
X_all = pd.concat([X_train, X_test], ignore_index=True)
y_all = pd.concat([y_train, y_test], ignore_index=True)
df_eda = X_all.copy()
df_eda['attack_category'] = y_all

print(f'✅ Données chargées')
print(f'   Train : {X_train.shape[0]:,} lignes × {X_train.shape[1]} features')
print(f'   Test  : {X_test.shape[0]:,} lignes × {X_test.shape[1]} features')
print(f'   Total EDA : {df_eda.shape[0]:,} lignes')
print(f'\nFeatures : {list(X_train.columns)}')

## 2. STATISTIQUES DESCRIPTIVES GLOBALES

In [ ]:
# ============================================
# STATISTIQUES DESCRIPTIVES
# ============================================

print('📈 STATISTIQUES DESCRIPTIVES GLOBALES')
print('-' * 60)
print(f'Nombre total de connexions : {len(df_eda):,}')
print(f'Nombre de features         : {X_train.shape[1]}')
print(f'Nombre de classes cibles   : {y_all.nunique()} ({list(y_all.unique())})')
print()

# Stats sur les features numériques clés
key_features = ['src_bytes_log', 'dst_bytes_log', 'duration_log',
                'count', 'srv_count', 'serror_rate', 'rerror_rate',
                'same_srv_rate', 'diff_srv_rate']

stats = X_all[key_features].describe().round(3)
print('Statistiques des features clés :')
print(stats)

# Valeurs manquantes
missing = df_eda.isnull().sum().sum()
print(f'\n✅ Valeurs manquantes totales : {missing}')

## 3. DISTRIBUTION DES CLASSES

In [ ]:
# ============================================
# DISTRIBUTION DES CLASSES
# ============================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Distribution des catégories d\'attaques (après préparation)', fontsize=14, fontweight='bold')

counts = y_all.value_counts().reindex(CLASS_ORDER)
colors = [CLASS_COLORS[c] for c in CLASS_ORDER]

# Barplot
bars = axes[0].bar(CLASS_ORDER, counts.values, color=colors, edgecolor='white', linewidth=1.2)
axes[0].set_title('Nombre de connexions par classe')
axes[0].set_xlabel('Catégorie')
axes[0].set_ylabel('Nombre de connexions')
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
                 f'{val:,}', ha='center', fontsize=10, fontweight='bold')

# Pie chart
axes[1].pie(counts.values, labels=CLASS_ORDER, colors=colors,
            autopct='%1.1f%%', startangle=140,
            wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
axes[1].set_title('Répartition en pourcentage')

plt.tight_layout()
plt.savefig('eda_01_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n📊 Résumé :')
for cat in CLASS_ORDER:
    n = counts[cat]
    print(f'   {cat:7}: {n:6,d}  ({n/len(y_all)*100:.1f}%)')

## 4. ANALYSE PAR PROTOCOLE

In [ ]:
# ============================================
# PROTOCOLE vs CLASSE D'ATTAQUE
# ============================================

# Reconstruire la colonne protocol depuis One-Hot
def get_protocol(row):
    if row['protocol_icmp'] == 1: return 'icmp'
    elif row['protocol_tcp'] == 1: return 'tcp'
    elif row['protocol_udp'] == 1: return 'udp'
    return 'unknown'

df_eda['protocol'] = df_eda.apply(get_protocol, axis=1)

# Tableau croisé
cross = pd.crosstab(df_eda['protocol'], df_eda['attack_category'])
cross = cross.reindex(columns=CLASS_ORDER, fill_value=0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Protocole réseau vs Catégorie d\'attaque', fontsize=14, fontweight='bold')

# Stacked bar
cross.plot(kind='bar', stacked=True, ax=axes[0],
           color=[CLASS_COLORS[c] for c in CLASS_ORDER],
           edgecolor='white', linewidth=0.5)
axes[0].set_title('Nombre de connexions (absolu)')
axes[0].set_xlabel('Protocole')
axes[0].set_ylabel('Nombre')
axes[0].legend(title='Classe', loc='upper right')
axes[0].tick_params(axis='x', rotation=0)

# Normalized stacked bar
cross_norm = cross.div(cross.sum(axis=1), axis=0) * 100
cross_norm.plot(kind='bar', stacked=True, ax=axes[1],
                color=[CLASS_COLORS[c] for c in CLASS_ORDER],
                edgecolor='white', linewidth=0.5)
axes[1].set_title('Proportion par protocole (%)')
axes[1].set_xlabel('Protocole')
axes[1].set_ylabel('Pourcentage')
axes[1].legend(title='Classe', loc='upper right')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig('eda_02_protocol_vs_class.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n📊 Tableau croisé (valeurs absolues) :')
print(cross)
print('\n📊 Tableau croisé (%) :')
print(cross_norm.round(1))

## 5. HEATMAP DE CORRÉLATION

In [ ]:
# ============================================
# HEATMAP DE CORRÉLATION
# ============================================

# Sélectionner les features les plus intéressantes
corr_features = [
    'src_bytes_log', 'dst_bytes_log', 'duration_log',
    'count', 'srv_count', 'serror_rate', 'srv_serror_rate',
    'rerror_rate', 'srv_rerror_rate', 'same_srv_rate', 'diff_srv_rate',
    'dst_host_count', 'dst_host_srv_count',
    'dst_host_same_srv_rate', 'dst_host_serror_rate',
    'logged_in_bin', 'flag_encoded'
]

corr_matrix = X_all[corr_features].corr()

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='RdYlGn', center=0, vmin=-1, vmax=1,
            ax=ax, linewidths=0.3, annot_kws={'size': 8},
            cbar_kws={'shrink': 0.8})
ax.set_title('Matrice de corrélation des features principales', fontsize=14, fontweight='bold', pad=15)
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.savefig('eda_03_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

# Top paires corrélées
print('\n🔝 Top 10 paires les plus corrélées :')
corr_pairs = corr_matrix.abs().unstack()
corr_pairs = corr_pairs[corr_pairs < 1.0].sort_values(ascending=False)
seen = set()
i = 0
for (a, b), val in corr_pairs.items():
    key = frozenset([a, b])
    if key not in seen:
        seen.add(key)
        print(f'   {a} ↔ {b}: {val:.3f}')
        i += 1
        if i >= 10: break

## 6. DISTRIBUTIONS DES FEATURES CLÉS PAR CLASSE

In [ ]:
# ============================================
# DISTRIBUTIONS PAR CLASSE (KDE + HIST)
# ============================================

features_to_plot = ['src_bytes_log', 'dst_bytes_log', 'serror_rate', 'same_srv_rate']
titles = [
    'log(1 + src_bytes) — Octets envoyés',
    'log(1 + dst_bytes) — Octets reçus',
    'serror_rate — Taux d\'erreurs SYN',
    'same_srv_rate — Taux même service'
]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Distribution des features clés par classe d\'attaque', fontsize=14, fontweight='bold')
axes = axes.flatten()

for idx, (feat, title) in enumerate(zip(features_to_plot, titles)):
    ax = axes[idx]
    for cat in CLASS_ORDER:
        subset = df_eda[df_eda['attack_category'] == cat][feat]
        if len(subset) > 1:
            subset.plot.kde(ax=ax, label=cat, color=CLASS_COLORS[cat], linewidth=2)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel(feat)
    ax.set_ylabel('Densité')
    ax.legend(title='Classe')
    ax.set_xlim([df_eda[feat].quantile(0.01), df_eda[feat].quantile(0.99)])

plt.tight_layout()
plt.savefig('eda_04_distributions_by_class.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. BOXPLOTS DES FEATURES LES PLUS DISCRIMINANTES

In [ ]:
# ============================================
# BOXPLOTS PAR CLASSE
# ============================================

box_features = ['src_bytes_log', 'dst_bytes_log', 'count', 'serror_rate']
box_titles = [
    'log(src_bytes)',
    'log(dst_bytes)',
    'count (connexions récentes)',
    'serror_rate'
]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Boxplots des features discriminantes par classe', fontsize=14, fontweight='bold')
axes = axes.flatten()

for idx, (feat, title) in enumerate(zip(box_features, box_titles)):
    ax = axes[idx]
    data = [df_eda[df_eda['attack_category'] == cat][feat].values for cat in CLASS_ORDER]
    bp = ax.boxplot(data, labels=CLASS_ORDER, patch_artist=True,
                    medianprops={'color': 'black', 'linewidth': 2},
                    whiskerprops={'linewidth': 1.2},
                    flierprops={'marker': 'o', 'markersize': 2, 'alpha': 0.3})
    for patch, cat in zip(bp['boxes'], CLASS_ORDER):
        patch.set_facecolor(CLASS_COLORS[cat])
        patch.set_alpha(0.75)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Catégorie')
    ax.set_ylabel(feat)

plt.tight_layout()
plt.savefig('eda_05_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. FEATURE IMPORTANCE PAR VARIANCE

In [ ]:
# ============================================
# FEATURE IMPORTANCE (VARIANCE INTER-CLASSES)
# ============================================

# Calculer la variance de la moyenne par classe pour chaque feature
numeric_features = [c for c in X_all.columns]
df_tmp = X_all.copy()
df_tmp['attack_category'] = y_all

class_means = df_tmp.groupby('attack_category')[numeric_features].mean()
inter_class_variance = class_means.var().sort_values(ascending=False)

top_n = 20
top_features = inter_class_variance.head(top_n)

fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.barh(range(top_n), top_features.values[::-1],
               color=sns.color_palette('Blues_r', top_n))
ax.set_yticks(range(top_n))
ax.set_yticklabels(top_features.index[::-1], fontsize=10)
ax.set_xlabel('Variance inter-classes (proxy d\'importance)', fontsize=11)
ax.set_title(f'Top {top_n} features les plus discriminantes', fontsize=14, fontweight='bold')
ax.invert_yaxis()

for bar, val in zip(bars, top_features.values[::-1]):
    ax.text(val + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('eda_06_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\n🔝 Top 10 features les plus discriminantes :')
for feat, val in inter_class_variance.head(10).items():
    print(f'   {feat:35}: {val:.4f}')

## 9. ANALYSE DES TAUX D'ERREUR PAR CLASSE

In [ ]:
# ============================================
# TAUX D'ERREUR MOYEN PAR CLASSE
# ============================================

rate_features = ['serror_rate', 'srv_serror_rate', 'rerror_rate',
                 'srv_rerror_rate', 'same_srv_rate', 'diff_srv_rate']

mean_by_class = df_eda.groupby('attack_category')[rate_features].mean().reindex(CLASS_ORDER)

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(rate_features))
width = 0.15

for i, cat in enumerate(CLASS_ORDER):
    ax.bar(x + i * width, mean_by_class.loc[cat], width,
           label=cat, color=CLASS_COLORS[cat], alpha=0.85, edgecolor='white')

ax.set_xticks(x + width * 2)
ax.set_xticklabels(rate_features, rotation=30, ha='right', fontsize=10)
ax.set_ylabel('Valeur moyenne (après normalisation)', fontsize=11)
ax.set_title('Taux d\'erreur et de service moyens par classe', fontsize=14, fontweight='bold')
ax.legend(title='Classe')
ax.axhline(0, color='grey', linewidth=0.8, linestyle='--')

plt.tight_layout()
plt.savefig('eda_07_error_rates_by_class.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n📊 Moyennes par classe :')
print(mean_by_class.round(3).to_string())

## 10. PROJECT SCHEMA — FLOW DIAGRAM

In [ ]:
# ============================================
# PROJECT SCHEMA (FLOW DIAGRAM)
# ============================================

fig, ax = plt.subplots(figsize=(14, 9))
ax.set_xlim(0, 14)
ax.set_ylim(0, 9)
ax.axis('off')
ax.set_facecolor('#F8F9FA')
fig.patch.set_facecolor('#F8F9FA')

# --- style helpers ---
def draw_box(ax, x, y, w, h, label, sublabel='', color='#4A90D9', text_color='white', radius=0.3):
    box = mpatches.FancyBboxPatch((x - w/2, y - h/2), w, h,
                                   boxstyle=f'round,pad={radius}',
                                   facecolor=color, edgecolor='white',
                                   linewidth=1.5, zorder=3)
    ax.add_patch(box)
    offset = 0.12 if sublabel else 0
    ax.text(x, y + offset, label, ha='center', va='center',
            fontsize=9, fontweight='bold', color=text_color, zorder=4)
    if sublabel:
        ax.text(x, y - 0.25, sublabel, ha='center', va='center',
                fontsize=7.5, color=text_color, alpha=0.9, zorder=4)

def draw_arrow(ax, x1, y1, x2, y2):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color='#555', lw=1.8), zorder=2)

# ---- TITRE ----
ax.text(7, 8.5, 'Network Intrusion Detection — KDD Cup 1999',
        ha='center', va='center', fontsize=14, fontweight='bold', color='#2C3E50')
ax.text(7, 8.1, 'Architecture globale du projet de Data Mining',
        ha='center', va='center', fontsize=10, color='#7F8C8D')

# ---- ROW 1: DATA SOURCE ----
draw_box(ax, 7, 7.2, 3.5, 0.7, 'KDD Cup 1999 (10%)', '494k connexions · 41 features', '#2C3E50')

draw_arrow(ax, 7, 6.85, 7, 6.45)

# ---- ROW 2: DATA PREPARATION (P2) ----
ax.text(7, 6.35, 'DATA PREPARATION (P2)', ha='center', fontsize=8, color='#27AE60', fontweight='bold')

prep_steps = [
    (3.5, 5.7, 'Dé-duplication', '494k → 145k (-70%)'),
    (5.8, 5.7, 'Encodage', 'OHE + LabelEncoder'),
    (8.2, 5.7, 'Outliers', 'log(1+x)'),
    (10.5, 5.7, 'Normalisation', 'StandardScaler'),
]
colors_prep = ['#27AE60', '#27AE60', '#27AE60', '#27AE60']
for (px, py, lbl, sub), col in zip(prep_steps, colors_prep):
    draw_box(ax, px, py, 2.0, 0.75, lbl, sub, col)

# Arrow connecting prep steps horizontally
for i in range(len(prep_steps) - 1):
    x1 = prep_steps[i][0] + 1.0
    x2 = prep_steps[i+1][0] - 1.0
    draw_arrow(ax, x1, 5.7, x2, 5.7)

# Arrow from source to first prep step
ax.annotate('', xy=(3.5, 6.08), xytext=(5.8, 6.38),
             arrowprops=dict(arrowstyle='->', color='#555', lw=1.4))

draw_arrow(ax, 10.5, 5.33, 10.5, 4.92)
ax.annotate('', xy=(7, 4.92), xytext=(10.5, 4.92),
             arrowprops=dict(arrowstyle='->', color='#555', lw=1.4))

# ---- REBALANCING ----
draw_box(ax, 7, 4.65, 3.0, 0.55, 'Rééquilibrage + Split 80/20', '43k lignes · Stratifié', '#8E44AD')
draw_arrow(ax, 7, 4.38, 7, 3.95)

# ---- ROW 3: EDA ----
ax.text(7, 3.82, 'EDA (P3)', ha='center', fontsize=8, color='#E67E22', fontweight='bold')
eda_steps = [
    (3.2, 3.25, 'Stats descriptives', 'mean/std/median'),
    (5.8, 3.25, 'Visualisations', 'Distrib. · Boxplots'),
    (8.4, 3.25, 'Corrélation', 'Heatmap'),
    (11.0, 3.25, 'Feature Importance', 'Variance inter-classes'),
]
for px, py, lbl, sub in eda_steps:
    draw_box(ax, px, py, 2.2, 0.75, lbl, sub, '#E67E22')

# Arrows from rebalancing down to EDA
for px, py, _, __ in eda_steps:
    ax.annotate('', xy=(px, py + 0.375), xytext=(7, 3.95),
                arrowprops=dict(arrowstyle='->', color='#BBB', lw=1))

# Arrows down from EDA to models
for px, py, _, __ in eda_steps:
    ax.annotate('', xy=(7, 2.38), xytext=(px, py - 0.375),
                arrowprops=dict(arrowstyle='->', color='#BBB', lw=1))

# ---- ROW 4: MODELS (P4) ----
ax.text(7, 2.25, 'MODEL BUILDING (P4)', ha='center', fontsize=8, color='#C0392B', fontweight='bold')
draw_box(ax, 4.2, 1.65, 3.2, 0.75, 'Decision Tree', 'Interprétable · max_depth', '#C0392B')
draw_box(ax, 9.8, 1.65, 3.2, 0.75, 'Random Forest', 'Ensemble · n_estimators', '#E74C3C')

# Arrows to evaluation
draw_arrow(ax, 4.2, 1.28, 5.5, 0.87)
draw_arrow(ax, 9.8, 1.28, 8.5, 0.87)

# ---- ROW 5: EVALUATION ----
draw_box(ax, 7, 0.65, 4.5, 0.65, 'Évaluation & Discussion (P4)',
         'Accuracy · F1 · Confusion Matrix · Meilleur modèle', '#1A252F')

# ---- LEGEND ----
legend_items = [
    ('#2C3E50', 'Source (P1)'),
    ('#27AE60', 'Préparation (P2)'),
    ('#E67E22', 'EDA (P3)'),
    ('#C0392B', 'Modèles (P4)'),
    ('#1A252F', 'Évaluation (P4)'),
]
for i, (col, lbl) in enumerate(legend_items):
    rx, ry = 0.3, 1.5 - i * 0.4
    ax.add_patch(mpatches.FancyBboxPatch((rx, ry - 0.12), 0.3, 0.24,
                  boxstyle='round,pad=0.05', facecolor=col, zorder=5))
    ax.text(rx + 0.45, ry, lbl, va='center', fontsize=8, color='#2C3E50')

plt.tight_layout()
plt.savefig('eda_08_project_schema.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Project Schema généré!')

## 11. RÉSUMÉ EDA

In [ ]:
# ============================================
# RÉSUMÉ FINAL
# ============================================

print('=' * 60)
print('📋 RÉSUMÉ DE L\'EDA — KDD Cup 1999')
print('=' * 60)

print(f'\n📦 Dataset (après préparation) :')
print(f'   {len(df_eda):,} connexions · {X_all.shape[1]} features')
print(f'   Train : {len(X_train):,}  |  Test : {len(X_test):,}')

print(f'\n📊 Distribution des classes :')
for cat in CLASS_ORDER:
    n = (y_all == cat).sum()
    print(f'   {cat:7}: {n:6,d} ({n/len(y_all)*100:.1f}%)')

print(f'\n🔍 Observations clés :')
print(f'   • DoS : fort serror_rate, src_bytes élevé → facilement détectable')
print(f'   • Probe : count et dst_host_count élevés → scan de ports')
print(f'   • R2L : logged_in_bin = 1 souvent, dst_bytes modéré')
print(f'   • U2R : très rare (52 exemples), patterns proches du normal')
print(f'   • Normal : same_srv_rate stable, rerror_rate faible')

print(f'\n📁 Visualisations générées :')
import os
for f in sorted(os.listdir('.')):
    if f.startswith('eda_') and f.endswith('.png'):
        print(f'   ✅ {f}')

print('\n✅ EDA terminée — prête pour le Model Building (P4)!')